In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/O.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed42_c.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_d4_seed2026_a_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/best_single-1.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/I.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed123_d_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/H.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/M.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/E.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submi

## Setup and Data Loading

In [2]:
import pandas as pd
import numpy as np
 
COMP = '/kaggle/input/competitions/playground-series-s6e4/'
DS   = '/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/'
 
sub  = pd.read_csv(COMP + 'sample_submission.csv')
 
a = pd.read_csv(DS + 'A.csv').rename(columns={'Irrigation_Need':'A'})
b = pd.read_csv(DS + 'B.csv').rename(columns={'Irrigation_Need':'B'})
c = pd.read_csv(DS + 'C.csv').rename(columns={'Irrigation_Need':'C'})
d = pd.read_csv(DS + 'D.csv').rename(columns={'Irrigation_Need':'D'})
x = pd.read_csv(DS + 'X.csv').rename(columns={'Irrigation_Need':'X'})
m = pd.read_csv(DS + 'M.csv').rename(columns={'Irrigation_Need':'M'})
n = pd.read_csv(DS + 'N.csv').rename(columns={'Irrigation_Need':'N'})
o = pd.read_csv(DS + 'O.csv').rename(columns={'Irrigation_Need':'O'})
p = pd.read_csv(DS + 'P.csv').rename(columns={'Irrigation_Need':'P'})
e = pd.read_csv(DS + 'E.csv').rename(columns={'Irrigation_Need':'E'})
f = pd.read_csv(DS + 'F.csv').rename(columns={'Irrigation_Need':'F'})
g = pd.read_csv(DS + 'G.csv').rename(columns={'Irrigation_Need':'G'})
h = pd.read_csv(DS + 'H.csv').rename(columns={'Irrigation_Need':'H'})
our = pd.read_csv(DS + 'submission_d4_s999.csv').rename(columns={'Irrigation_Need':'OUR'})
 

## Merge all

In [3]:
dfs = (a.merge(b,on='id').merge(c,on='id').merge(d,on='id')
        .merge(x,on='id').merge(m,on='id').merge(n,on='id').merge(o,on='id').merge(p,on='id')
        .merge(e,on='id').merge(f,on='id').merge(g,on='id').merge(h,on='id').merge(our,on='id'))
 
dfs['abcd_agree']  = dfs[['A','B','C','D']].nunique(axis=1) == 1
dfs['no_agree']    = dfs['N'] == dfs['O']
dfs['eho_agree']   = dfs[['E','H','OUR']].nunique(axis=1) == 1
dfs['top5_maj']    = dfs[['E','H','OUR','G','N']].apply(
    lambda r: max(set(r), key=list(r).count), axis=1)

## Submissions

In [4]:
combos = {
    'S1_130k_N_EHOmaj': (130000, 'top5_maj', 'N'),
    'S2_137k_N_EHOmaj': (137400, 'top5_maj', 'N'),
    'S3_140k_N_EHOmaj': (140000, 'top5_maj', 'N'),
    'S4_137k_H_OUR':    (137400, 'H',         'N'),   
    'S5_137k_E_OUR':    (137400, 'E',         'N'), 
    'S6_137k_H_E':      (137400, 'H',         'E'),   
    'S7_135k_N_P':      (135000, 'P',         'N'),   
}
 
def make_submission(sl, agreed_src, disagreed_src):
    """
    sl: slice point
    agreed_src: column to use when N==O (rows sl onwards)
    disagreed_src: column to use when N!=O (rows sl onwards)
    """
    final = []
    for idx in range(len(dfs)):
        if idx < sl:
            final.append(dfs['N'].iloc[idx])
        else:
            if dfs['N'].iloc[idx] == dfs['O'].iloc[idx]:
                final.append(dfs[agreed_src].iloc[idx])
            else:
                final.append(dfs[disagreed_src].iloc[idx])
    return final
 
results = {}
for name, (sl, agreed, disagreed) in combos.items():
    final = make_submission(sl, agreed, disagreed)
    s = pd.Series(final)
    out = sub.copy()
    out['Irrigation_Need'] = final
    fname = f'submission_{name}.csv'
    out.to_csv(fname, index=False)
    dist = out['Irrigation_Need'].value_counts().to_dict()
    print(f"{name:35s}  High={dist.get('High',0):,}  → {fname}")
 
# full EHO majority
sub_eho_full = sub.copy()
sub_eho_full['Irrigation_Need'] = dfs.apply(
    lambda r: r['E'] if r['eho_agree'] else r['top5_maj'], axis=1)
sub_eho_full.to_csv('submission_full_EHO.csv', index=False)
dist = sub_eho_full['Irrigation_Need'].value_counts().to_dict()
print(f"{'full_EHO':35s}  High={dist.get('High',0):,}  → submission_full_EHO.csv")
 
# Best + majority of E,H,OUR on disagreements
def nina_ehomaj(row, idx):
    if idx < 137400:
        return row['N']
    if row['N'] == row['O']:
        return row['E'] if row['eho_agree'] else row['top5_maj']
    else:
        return row['top5_maj']
 
sub_nina_ehomaj = sub.copy()
sub_nina_ehomaj['Irrigation_Need'] = [
    nina_ehomaj(dfs.iloc[i], i) for i in range(len(dfs))]
sub_nina_ehomaj.to_csv('submission_Best_EHOmaj.csv', index=False)
dist = sub_nina_ehomaj['Irrigation_Need'].value_counts().to_dict()
print(f"{'Best_EHOmaj':35s}  High={dist.get('High',0):,}  → submission_Best_EHOmaj.csv")
 
print("\n" + "="*60)

S1_130k_N_EHOmaj                     High=10,234  → submission_S1_130k_N_EHOmaj.csv
S2_137k_N_EHOmaj                     High=10,232  → submission_S2_137k_N_EHOmaj.csv
S3_140k_N_EHOmaj                     High=10,232  → submission_S3_140k_N_EHOmaj.csv
S4_137k_H_OUR                        High=10,242  → submission_S4_137k_H_OUR.csv
S5_137k_E_OUR                        High=10,252  → submission_S5_137k_E_OUR.csv
S6_137k_H_E                          High=10,242  → submission_S6_137k_H_E.csv
S7_135k_N_P                          High=9,597  → submission_S7_135k_N_P.csv
full_EHO                             High=10,206  → submission_full_EHO.csv
Best_EHOmaj                          High=10,227  → submission_Best_EHOmaj.csv

